# Cuaderno 02: Modelado Supervisado
**Objetivo:** Implementar y entrenar modelos de regresión para predecir el precio de las propiedades en California. [cite_start]Se utilizarán **pipelines de Scikit-learn** para integrar de forma segura el preprocesamiento final y el entrenamiento de los algoritmos[cite: 104].

## 

*1. Configuración y Carga de Datos Procesados*

En esta etapa cargamos el dataset que ya ha sido limpiado y filtrado en el Cuaderno 01. Esto nos permite separar la fase de exploración de la fase de modelado, asegurando la modularidad del proyecto[cite: 29].

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import joblib
import os

# Cargamos el dataset YA LIMPIO generado en el Cuaderno 01
df = pd.read_csv('../data/RealEstate_California_Clean.csv')

# Seleccionamos las columnas que utilizaremos como predictores
# Basado en la correlación observada en el EDA
features_num = ['livingArea', 'bathrooms', 'bedrooms', 'longitude', 'latitude', 'yearBuilt']
features_cat = ['homeType', 'hasGarage']

# Definimos la variable objetivo (target)
target = 'price'

# Separamos las características (X) de la etiqueta (y)
X = df[features_num + features_cat]
y = df[target]

print(f"Dataset cargado exitosamente con {X.shape[0]} registros.")

Dataset cargado exitosamente con 34420 registros.


*2. División de Datos (Train / Test Split)*
Para garantizar una evaluación justa y evitar el *data leakage*, separamos el conjunto de datos en un 80% para entrenamiento y un 20% para prueba.

**Justificación técnica:** Se utiliza un `random_state` fijo para asegurar que los resultados sean **100% reproducibles**, tal como lo exige la rúbrica de evaluación[cite: 59].

In [3]:
# División de datos (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Set de entrenamiento: {X_train.shape}")
print(f"Set de prueba: {X_test.shape}")

Set de entrenamiento: (27536, 8)
Set de prueba: (6884, 8)


*3. Construcción del Pipeline de Preprocesamiento*

Diseñamos un `ColumnTransformer` para manejar distintos tipos de datos de forma automatizada:
**Numéricas:** Aplicamos `KNNImputer` para manejar valores nulos basándonos en la similitud de vecinos y `StandardScaler` para normalizar las escalas[cite: 104, 109].
* **Categóricas:** Usamos `SimpleImputer` (frecuencia) y `OneHotEncoder` para convertir categorías en vectores numéricos.

**Justificación técnica:** El uso de **pipelines** previene que información del set de prueba se filtre durante la imputación o el escalado[cite: 25].

In [4]:
# Definición de pasos para datos numéricos
num_pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler())
])

# Definición de pasos para datos categóricos
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Integración en un transformador de columnas
preprocessor = ColumnTransformer([
    ('num', num_pipeline, features_num),
    ('cat', cat_pipeline, features_cat)
])

print("Pipeline de preprocesamiento configurado.")

Pipeline de preprocesamiento configurado.


*4. Definición y Entrenamiento de Modelos Supervisados*

Implementamos dos tipos de modelos para comparar su rendimiento:
1.  **Regresión Lineal:** Modelo base para establecer un punto de referencia (baseline).
2.  **Random Forest Regressor:** Modelo de ensamble capaz de capturar relaciones no lineales y complejas[cite: 104].

In [5]:
# 1. Pipeline para Regresión Lineal
pipeline_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# 2. Pipeline para Random Forest
pipeline_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

# Entrenamiento de los modelos
print("Iniciando entrenamiento de Regresión Lineal...")
pipeline_lr.fit(X_train, y_train)

print("Iniciando entrenamiento de Random Forest (puede demorar)...")
pipeline_rf.fit(X_train, y_train)

print("¡Modelos entrenados correctamente!")

Iniciando entrenamiento de Regresión Lineal...
Iniciando entrenamiento de Random Forest (puede demorar)...
¡Modelos entrenados correctamente!


*5. Serialización y Persistencia del Modelo*

Para cumplir con la estructura de archivos solicitada, guardamos los modelos entrenados y los datos divididos en disco. [cite_start]Esto permitirá su posterior evaluación u optimización en cuadernos independientes[cite: 29, 41].

In [6]:
# Aseguramos que existan las carpetas necesarias
os.makedirs('../models/trained_models/', exist_ok=True)

# Guardamos los modelos (serialización)
joblib.dump(pipeline_lr, '../models/trained_models/pipeline_lr.pkl')
joblib.dump(pipeline_rf, '../models/trained_models/pipeline_rf.pkl')

# Guardamos los datos divididos para mantener la consistencia en el siguiente cuaderno
joblib.dump((X_train, X_test, y_train, y_test), '../data/train_test_data.pkl')

print("Modelos y datos guardados exitosamente en la carpeta raíz del proyecto.")

Modelos y datos guardados exitosamente en la carpeta raíz del proyecto.
